# Evaluation of community crime classifier

This notebook performs a detailed evaluation of the trained classifiers:
classification reports, per-class precision and recall, feature importance,
misclassification analysis, and a discussion of how results could feed
into a map-based dashboard.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, precision_recall_fscore_support,
)

from src.data_loader import (
    load_or_fetch_crime_data, load_or_fetch_census_data,
    preprocess_crime_data, preprocess_census_data,
    create_community_features,
)
from src.model import (
    create_risk_labels, prepare_classification_data,
    train_classifiers, get_feature_importance,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Rebuild models for evaluation

We reload the community dataset and retrain so that we have access to both
the models and the test set metadata for error analysis.

In [ ]:
import os

engineered_path = '../data/community_features_engineered.csv'
if os.path.exists(engineered_path):
    community_df = pd.read_csv(engineered_path)
else:
    crime_raw = load_or_fetch_crime_data('../data')
    census_raw = load_or_fetch_census_data('../data')
    crime_df = preprocess_crime_data(crime_raw)
    census_df = preprocess_census_data(census_raw)
    community_df = create_community_features(crime_df, census_df)

if 'risk_level' not in community_df.columns:
    community_df = create_risk_labels(community_df)

X, y, label_encoder, feature_cols = prepare_classification_data(community_df)
trained_models, results, scaler, X_test, y_test = train_classifiers(X, y, random_state=42)

class_names = list(label_encoder.classes_)
print(f'Evaluation on {len(X_test)} test communities')
print(f'Classes: {class_names}')

## 2. Full classification report

The classification report shows precision, recall, and F1 for each class
along with macro and weighted averages.

In [ ]:
best_name = max(results, key=lambda k: results[k]['Accuracy'])
best_model = trained_models[best_name]

if best_name == 'Logistic Regression':
    y_pred = best_model.predict(scaler.transform(X_test))
else:
    y_pred = best_model.predict(X_test)

print(f'Classification report for {best_name}:\n')
print(classification_report(y_test, y_pred, target_names=class_names))

## 3. Per-class precision and recall bar chart

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=range(len(class_names))
)

metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1': f1,
    'Support': support,
})
print(metrics_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(class_names))
width = 0.25
ax.bar(x - width, precision, width, label='Precision', color='#3498db')
ax.bar(x, recall, width, label='Recall', color='#e74c3c')
ax.bar(x + width, f1, width, label='F1', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_ylabel('Score')
ax.set_title(f'Per-class metrics -- {best_name}', fontsize=12)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## 4. Feature importance

We extract feature importances from the tree-based best model using
`get_feature_importance` from `model.py`.

In [ ]:
importance_df = get_feature_importance(best_model, feature_cols)

if not importance_df.empty:
    fig, ax = plt.subplots(figsize=(8, 6))
    top_features = importance_df.head(15)
    ax.barh(top_features['Feature'], top_features['Importance'],
            color='steelblue', edgecolor='black')
    ax.invert_yaxis()
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 15 features -- {best_name}', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(importance_df.head(15).to_string(index=False))
else:
    print('Feature importances not available for this model type.')

## 5. Misclassification analysis

We identify which communities were misclassified and examine their
characteristics to understand the error patterns.

In [ ]:
# Build evaluation dataframe
eval_df = community_df.loc[X_test.index].copy()
eval_df['actual'] = label_encoder.inverse_transform(y_test)
eval_df['predicted'] = label_encoder.inverse_transform(y_pred)
eval_df['correct'] = eval_df['actual'] == eval_df['predicted']

misclassified = eval_df[~eval_df['correct']]
print(f'Misclassified communities: {len(misclassified)} out of {len(eval_df)}')
print(f'Accuracy: {eval_df["correct"].mean():.2%}\n')

if len(misclassified) > 0:
    display_cols = ['community', 'total_crimes', 'avg_monthly_crimes',
                    'actual', 'predicted']
    available_display = [c for c in display_cols if c in misclassified.columns]
    print('Misclassified communities:')
    print(misclassified[available_display].to_string(index=False))

In [ ]:
# Where does confusion happen?
if len(misclassified) > 0:
    confusion_pairs = misclassified.groupby(['actual', 'predicted']).size().reset_index(name='count')
    print('\nConfusion pairs:')
    print(confusion_pairs.sort_values('count', ascending=False).to_string(index=False))

## 6. All-models comparison

A consolidated view of accuracy and F1 across all trained classifiers.

In [ ]:
comparison = pd.DataFrame(results).T.round(4)
comparison = comparison.sort_values('Accuracy', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
comparison[['Accuracy', 'F1 (Weighted)', 'F1 (Macro)']].plot(
    kind='bar', ax=ax, edgecolor='black'
)
ax.set_title('Classifier comparison', fontsize=12)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=25)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(comparison.to_string())

## 7. Map visualization discussion

While this notebook does not render an interactive map (it requires a running
Jupyter server with folium or plotly), the predicted risk levels can be
directly joined to Calgary community boundary shapefiles to produce a
chloropleth map. The workflow would be:

1. Load community boundary GeoJSON from the Calgary Open Data portal.
2. Join `community_df[['community', 'risk_level']]` on the community name.
3. Colour each polygon by risk level (green / yellow / red).
4. Add hover tooltips with crime statistics and model confidence.

This map would be a useful tool for city planners to allocate policing
resources and community safety programs.

In [ ]:
# Example: prepare the data that would feed into a folium map
map_data = community_df[['community', 'risk_level', 'total_crimes', 'avg_monthly_crimes']].copy()
map_data['risk_color'] = map_data['risk_level'].map({
    'Low': 'green', 'Medium': 'orange', 'High': 'red'
})
print(f'Map data prepared: {len(map_data)} communities')
print(map_data.head(10).to_string(index=False))

## Summary

- The best classifier achieves high accuracy on the three-class risk prediction
  task, with ensemble methods outperforming simpler baselines.
- The Medium class is hardest to classify because it sits at the boundary
  between Low and High, and its crime statistics overlap with both.
- Feature importance shows that aggregate crime counts and specific category
  breakdowns (theft, disorder) are the strongest predictors.
- Misclassified communities tend to have crime counts near the percentile
  cutoff boundaries, which is an inherent limitation of discretising a
  continuous distribution into three bins.
- Results are ready for integration into a map-based community safety dashboard.